# Level 4, Notebook 2: Retries, dead letters, and timeouts

**Problems in this notebook**

- Problem 32: Retry Failed API Calls with Exponential Backoff
- Problem 36: Dead-Letter Queue Handling
- Problem 38: Model Response Timeout Handling

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `02_retries_and_failures_THEORY.md` in this folder.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 32: Retry Failed API Calls with Exponential Backoff

**The situation**

Model APIs and vector stores sometimes fail for a moment and then work again. Retrying helps, but retrying too fast or forever makes an outage worse for everyone.

**Your task**

Write `retry_with_backoff(fn, max_attempts=3, base_delay=0.5, sleep=time.sleep, retryable=(TimeoutError, ConnectionError))`. Retry only retryable errors, wait `base_delay * 2**(attempt - 1)` between tries, and give up after `max_attempts`. `sleep` is passed in so tests run instantly.

**Example**

A call that raises `TimeoutError` once and then returns `"ok"` gives `"ok"`, after one wait of `base_delay`. A `ValueError` is raised right away with no retry.

**Ask before coding**

- Which errors are temporary and safe to retry?
- Is the operation safe to repeat (a read), or could a retry do something twice (a write)?
- What is the most attempts and the longest total wait we accept?

**Plan to say aloud**

1. Loop from attempt 1 to `max_attempts` and call `fn()`.
2. On success, return the result.
3. On a non-retryable error, raise it at once.
4. On a retryable error, raise it if this was the last attempt. Otherwise sleep and try again, doubling the wait.

### Prompt for your AI assistant

```text
Write a Python function retry_with_backoff(fn, max_attempts: int = 3, base_delay: float = 0.5, sleep=time.sleep, retryable: tuple = (TimeoutError, ConnectionError)). Call fn() up to max_attempts times. Return its result on success. Raise non-retryable exceptions immediately. Between attempts call sleep(base_delay * 2 ** (attempt - 1)). Do not sleep after the last attempt. After the last failure, re-raise that exception. Use the sleep argument, never time.sleep directly. Show only the function.
```

**Check the AI's code for**

- Catches every `Exception`, so a bad request (`ValueError`) is retried three times for nothing.
- Calls `time.sleep` directly instead of the `sleep` argument, so tests really wait.
- Sleeps once more after the final failed attempt.
- Swallows the last error and returns `None` instead of raising it.

In [ ]:
import time

def retry_with_backoff(fn, max_attempts: int = 3, base_delay: float = 0.5, sleep=time.sleep,
                       retryable: tuple = (TimeoutError, ConnectionError)):
    """Call fn() and retry temporary failures with exponential backoff.

    Rules:
    - return fn()'s result as soon as it succeeds
    - non-retryable errors are raised at once
    - wait base_delay * 2 ** (attempt - 1) between attempts, using sleep()
    - never sleep after the last attempt; re-raise the last error
    """
    raise NotImplementedError

In [ ]:
def simulate_retry(retry_fn, outcomes, max_attempts):
    """outcomes: what each call does, in order. An exception is raised, anything else is returned.
    Returns (result or error name, list of waits)."""
    calls = iter(outcomes)
    waits = []
    def flaky():
        item = next(calls)
        if isinstance(item, Exception):
            raise item
        return item
    try:
        result = retry_fn(flaky, max_attempts=max_attempts, base_delay=1.0, sleep=waits.append)
    except NotImplementedError:
        raise
    except Exception as e:
        result = type(e).__name__
    return result, waits

cases = [
    (([TimeoutError(), "ok"], 3), ("ok", [1.0])),
    ((["ok"], 3), ("ok", [])),
    (([TimeoutError(), ConnectionError(), "ok"], 3), ("ok", [1.0, 2.0])),
    (([TimeoutError(), TimeoutError(), TimeoutError()], 3), ("TimeoutError", [1.0, 2.0])),  # give up, no last wait
    (([ValueError("bad prompt"), "ok"], 3), ("ValueError", [])),                              # not retryable
]
run_checks(lambda *a: simulate_retry(retry_with_backoff, *a), cases)

### Reference solution

Try first. Then compare.

In [ ]:
import time

def retry_with_backoff_reference(fn, max_attempts: int = 3, base_delay: float = 0.5, sleep=time.sleep,
                                 retryable: tuple = (TimeoutError, ConnectionError)):
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except retryable:
            if attempt == max_attempts:
                raise
            sleep(base_delay * 2 ** (attempt - 1))

assert run_checks(lambda *a: simulate_retry(retry_with_backoff_reference, *a), cases)

**Explain it in 30 seconds**

> "I call the function in a loop with a fixed number of attempts. Only temporary errors like timeouts are retried; anything else goes straight back to the caller. Each wait doubles, so a struggling service gets room to recover. On the last attempt I re-raise the error so nothing fails silently."

**Edge cases to mention**

- A write that is not safe to repeat, such as sending an email or creating a ticket.
- A rate-limit error that tells you how long to wait (respect that instead of your own delay).
- Many clients retrying at the same moment, which is why real systems add random jitter.
- `max_attempts=1`, which means no retries at all.

**If they ask "how would this work in production?"**

- Add jitter (a small random extra wait) so thousands of clients do not retry in sync.
- Use idempotency keys on writes so a retried request is not applied twice.
- Cap total retries per service (a retry budget) and add a circuit breaker that stops calling a service that is clearly down.

## Problem 36: Dead-Letter Queue Handling

**The situation**

A document ingestion worker keeps failing on some files. A broken file retried forever wastes money and can block the jobs behind it. After a few tries it should be parked in a dead-letter queue (DLQ) for a human to look at.

**Your task**

Write `route_failed_job(job, max_attempts)` that returns `"retry"` or `"dead_letter"`. Park the job when it hit the attempt limit, when its error is marked not retryable, or when its attempt count is missing or broken.

**Example**

`{"attempts": 3}` with `max_attempts=3` gives `"dead_letter"`. `{"attempts": 1}` gives `"retry"`. `{"attempts": 0, "retryable": False}` gives `"dead_letter"`.

**Ask before coding**

- Does `attempts` count tries already made?
- Do we know which errors are permanent, such as a corrupt file?
- If the attempt count is missing, should we retry or park the job?

**Plan to say aloud**

1. Read `attempts`. If it is not a whole number of zero or more, park the job.
2. If the job says its error is not retryable, park it.
3. If attempts reached `max_attempts`, park it.
4. Otherwise send it back for a retry.

### Prompt for your AI assistant

```text
Write a Python function route_failed_job(job: dict, max_attempts: int) -> str that returns 'retry' or 'dead_letter'. Rules: attempts = job.get('attempts'); if it is not an int or is negative, return 'dead_letter'. If job.get('retryable', True) is False, return 'dead_letter'. If attempts >= max_attempts, return 'dead_letter'. Otherwise return 'retry'. Never raise on a malformed job. Show only the function.
```

**Check the AI's code for**

- Defaults a missing `attempts` to 0, so a job whose counter is never saved retries forever.
- Uses `>` instead of `>=`, which allows one extra attempt.
- Ignores the `retryable` flag and retries a corrupt file several times.
- Calls `int(job['attempts'])`, which crashes on a missing key or bad value.

In [ ]:
def route_failed_job(job: dict, max_attempts: int) -> str:
    """Return "retry" or "dead_letter" for a failed job.

    Rules:
    - attempts missing, not an int, or negative -> "dead_letter"
    - job["retryable"] is False -> "dead_letter"
    - attempts >= max_attempts -> "dead_letter"
    - otherwise -> "retry"
    """
    raise NotImplementedError

In [ ]:
cases = [
    (({"id": "d1", "attempts": 3}, 3), "dead_letter"),
    (({"id": "d2", "attempts": 1}, 3), "retry"),
    (({"id": "d3", "attempts": 0, "retryable": False}, 3), "dead_letter"),  # corrupt file: no point retrying
    (({"id": "d4"}, 3), "dead_letter"),                                     # no counter: cannot track it
    (({"id": "d5", "attempts": -1}, 3), "dead_letter"),
    (({"id": "d6", "attempts": 2, "retryable": True}, 3), "retry"),
]
run_checks(route_failed_job, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def route_failed_job_reference(job: dict, max_attempts: int) -> str:
    attempts = job.get("attempts")
    if not isinstance(attempts, int) or attempts < 0:
        return "dead_letter"
    if job.get("retryable", True) is False:
        return "dead_letter"
    if attempts >= max_attempts:
        return "dead_letter"
    return "retry"

assert run_checks(route_failed_job_reference, cases)

**Explain it in 30 seconds**

> "I park a job in the dead-letter queue in three cases: it used up its attempts, its error is permanent, or its attempt count is broken so I cannot track it. Everything else goes back for another try. The point is that no job loops forever, and nothing is thrown away: the DLQ keeps it so someone can fix and replay it."

**Edge cases to mention**

- Missing or negative attempt count.
- A poison message that crashes the worker every single time.
- A permanent error on the first try (no point using all attempts).
- A very large payload, where you store a pointer to the file rather than the file itself.

**If they ask "how would this work in production?"**

- Store the original payload, error reason, tenant id, and retry history with each DLQ entry.
- Alert when the DLQ grows, and give the team a tool to replay fixed jobs.
- Use the queue's own redelivery count (most brokers track it) rather than trusting a field in the message.

## Problem 38: Model Response Timeout Handling

**The situation**

An enterprise contract says answers must arrive within 10 seconds. Some model calls take longer, and some stream part of an answer before running out of time. The app needs a clear label for each call to decide on a fallback.

**Your task**

Write `classify_model_call(latency_ms, timeout_ms, streamed_tokens=0)` that returns `"success"`, `"partial"`, `"timeout"`, or `"invalid"`.

**Example**

`(12000, 10000)` gives `"timeout"`. `(10000, 10000)` gives `"success"`. `(12000, 10000, 40)` gives `"partial"`. `(-5, 10000)` gives `"invalid"`.

**Ask before coding**

- Does the time limit include retrieval, or only the model call?
- If part of the answer streamed in time, does that count?
- Is a call that takes exactly the limit a success?

**Plan to say aloud**

1. If latency is negative or the limit is not positive, return `"invalid"`.
2. If latency is within the limit (equal counts), return `"success"`.
3. Over the limit: if some tokens streamed, return `"partial"`, else `"timeout"`.

### Prompt for your AI assistant

```text
Write a Python function classify_model_call(latency_ms: float, timeout_ms: float, streamed_tokens: int = 0) -> str. Return 'invalid' if latency_ms < 0 or timeout_ms <= 0. Return 'success' if latency_ms <= timeout_ms (equal is a success). Otherwise return 'partial' if streamed_tokens > 0, else 'timeout'. Show only the function.
```

**Check the AI's code for**

- Uses `>=` so a call that lands exactly on the limit counts as a timeout.
- Accepts negative latency, which only happens with a clock bug, and labels it a success.
- Ignores `streamed_tokens`, so a half-streamed answer looks the same as no answer.
- Accepts a zero or negative limit, which marks every call as a timeout.

In [ ]:
def classify_model_call(latency_ms: float, timeout_ms: float, streamed_tokens: int = 0) -> str:
    """Label a finished model call.

    Rules:
    - latency_ms < 0 or timeout_ms <= 0 -> "invalid"
    - latency_ms <= timeout_ms -> "success" (equal counts as success)
    - over the limit with streamed_tokens > 0 -> "partial"
    - over the limit with nothing streamed -> "timeout"
    """
    raise NotImplementedError

In [ ]:
cases = [
    ((12000, 10000), "timeout"),
    ((10000, 10000), "success"),     # exactly on the limit
    ((800, 10000), "success"),
    ((12000, 10000, 40), "partial"), # some text reached the user in time
    ((-5, 10000), "invalid"),        # clock bug
    ((500, 0), "invalid"),           # bad config
]
run_checks(classify_model_call, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def classify_model_call_reference(latency_ms: float, timeout_ms: float, streamed_tokens: int = 0) -> str:
    if latency_ms < 0 or timeout_ms <= 0:
        return "invalid"
    if latency_ms <= timeout_ms:
        return "success"
    return "partial" if streamed_tokens > 0 else "timeout"

assert run_checks(classify_model_call_reference, cases)

**Explain it in 30 seconds**

> "First I reject impossible inputs, like negative latency, so a clock bug does not look like a fast call. Within the limit is a success, and exactly on the limit still counts. Over the limit, I separate a partial answer from no answer, because the fallback is different: keep the partial text or switch models."

**Edge cases to mention**

- Latency exactly equal to the limit.
- Negative latency from a clock bug.
- A stream that started on time but did not finish.
- Different tenants with different time limits in their contracts.

**If they ask "how would this work in production?"**

- Labelling after the fact is not enough: enforce the limit while the call runs (for example `asyncio.wait_for`) and cancel it.
- On timeout, fall back to a faster or cheaper model, but count the extra cost and latency.
- Trace latency per step (retrieval, model, post-processing) so you know which part is slow.

## Wrap-up

- Not every failure deserves a retry. Sort errors into temporary and permanent first.
- Every retry loop needs an end: a maximum number of attempts, then a dead-letter queue or a clear error.
- Inject `sleep` and pass in measured latency so failure handling can be tested instantly, without real waiting.